# Zomato Dataset — Data Cleaning & Preprocessing

## 1. Objective

The objective of this notebook is to clean and preprocess the raw Zomato dataset so that it can be used for exploratory analysis, feature engineering, and machine learning.

## 2. Cleaning Tasks

The following preprocessing steps will be performed:

- Load the raw dataset
- Standardize column names
- Remove irrelevant columns
- Handle missing values
- Clean restaurant ratings
- Convert cost into numerical format
- Clean categorical variables
- Handle inconsistent text values
- Check for duplicate records
- Validate the cleaned dataset
- Save the processed dataset

## 3. Important Principles

The raw dataset will not be modified directly.

All transformations will be performed on a copy of the raw data, and the cleaned dataset will be saved separately.

Raw data:
`data/raw/zomato.csv`

Processed data:
`data/processed/zomato_cleaned.csv`

## 4. Expected Outcome

The final cleaned dataset should:

- Contain consistent data types
- Have appropriately handled missing values
- Have numerical variables in usable numerical format
- Have standardized categorical values
- Remove columns that provide no predictive value
- Be ready for exploratory analysis and feature engineering

In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv(r"C:\Users\USER\OneDrive\Desktop\Business_Analysis_Agent\data\raw\zomato.csv")

print("Dataset shape:", df.shape)
display(df.head())

Dataset shape: (51717, 17)


,url,address,name,online_order,book_table,rate,votes,phone,location,rest_type,dish_liked,cuisines,approx_cost(for two people),reviews_list,menu_item,listed_in(type),listed_in(city)
0,https://www.zomato.com/bangalore/jalsa-banasha...,"942, 21st Main Road, 2nd Stage, Banashankari, ...",Jalsa,Yes,Yes,4.1/5,775,080 42297555\r\n+91 9743772233,Banashankari,Casual Dining,"Pasta, Lunch Buffet, Masala Papad, Paneer Laja...","North Indian, Mughlai, Chinese",800,"[('Rated 4.0', 'RATED\n A beautiful place to ...",[],Buffet,Banashankari
1,https://www.zomato.com/bangalore/spice-elephan...,"2nd Floor, 80 Feet Road, Near Big Bazaar, 6th ...",Spice Elephant,Yes,No,4.1/5,787,080 41714161,Banashankari,Casual Dining,"Momos, Lunch Buffet, Chocolate Nirvana, Thai G...","Chinese, North Indian, Thai",800,"[('Rated 4.0', 'RATED\n Had been here for din...",[],Buffet,Banashankari
2,https://www.zomato.com/SanchurroBangalore?cont...,"1112, Next to KIMS Medical College, 17th Cross...",San Churro Cafe,Yes,No,3.8/5,918,+91 9663487993,Banashankari,"Cafe, Casual Dining","Churros, Cannelloni, Minestrone Soup, Hot Choc...","Cafe, Mexican, Italian",800,"[('Rated 3.0', ""RATED\n Ambience is not that ...",[],Buffet,Banashankari
3,https://www.zomato.com/bangalore/addhuri-udupi...,"1st Floor, Annakuteera, 3rd Stage, Banashankar...",Addhuri Udupi Bhojana,No,No,3.7/5,88,+91 9620009302,Banashankari,Quick Bites,Masala Dosa,"South Indian, North Indian",300,"[('Rated 4.0', ""RATED\n Great food and proper...",[],Buffet,Banashankari
4,https://www.zomato.com/bangalore/grand-village...,"10, 3rd Floor, Lakshmi Associates, Gandhi Baza...",Grand Village,No,No,3.8/5,166,+91 8026612447\r\n+91 9901210005,Basavanagudi,Casual Dining,"Panipuri, Gol Gappe","North Indian, Rajasthani",600,"[('Rated 4.0', 'RATED\n Very good restaurant ...",[],Buffet,Banashankari


In [2]:
df_clean = df.copy()

print("Working copy created.")
print("Shape:", df_clean.shape)

Working copy created.
Shape: (51717, 17)


## Standardize Column Names

Column names are standardized to lowercase and simplified so that they are easier to work with during preprocessing and feature engineering.

In [3]:
df_clean.columns = (
    df_clean.columns
    .str.lower()
    .str.strip()
    .str.replace(" ", "_")
    .str.replace("(", "", regex=False)
    .str.replace(")", "", regex=False)
    .str.replace("/", "_", regex=False)
)

print(df_clean.columns.tolist())

['url', 'address', 'name', 'online_order', 'book_table', 'rate', 'votes', 'phone', 'location', 'rest_type', 'dish_liked', 'cuisines', 'approx_costfor_two_people', 'reviews_list', 'menu_item', 'listed_intype', 'listed_incity']


Remove Irrelevant Columns

In [4]:
df_clean.drop(
    columns=["url", "phone"],
    inplace=True
)

print(df_clean.shape)
print(df_clean.columns.tolist())

(51717, 15)
['address', 'name', 'online_order', 'book_table', 'rate', 'votes', 'location', 'rest_type', 'dish_liked', 'cuisines', 'approx_costfor_two_people', 'reviews_list', 'menu_item', 'listed_intype', 'listed_incity']


Clean Restaurant Ratings

The `rate` column contains values such as:

- `4.1/5`
- `3.8 /5`
- `NEW`
- Missing values

The rating will be converted into a numerical value between 0 and 5.

Restaurants marked as `NEW` do not have an established rating and will therefore be treated as missing values rather than as a rating of zero.

In [5]:
df_clean["rate"] = (
    df_clean["rate"]
    .replace("NEW", np.nan)
    .str.replace("/5", "", regex=False)
    .str.strip()
)

df_clean["rate"] = pd.to_numeric(
    df_clean["rate"],
    errors="coerce"
)

print(df_clean["rate"].dtype)
print(df_clean["rate"].describe())
print("\nMissing ratings:", df_clean["rate"].isna().sum())

float64
count    41665.000000
mean         3.700449
std          0.440513
min          1.800000
25%          3.400000
50%          3.700000
75%          4.000000
max          4.900000
Name: rate, dtype: float64

Missing ratings: 10052


Clean Approximate Cost

The `approx_cost_for_two_people` column contains values stored as strings, including commas such as `1,000`.

The column will be converted into a numerical variable representing the approximate cost for two people.

In [6]:
print(df_clean.columns.tolist())

['address', 'name', 'online_order', 'book_table', 'rate', 'votes', 'location', 'rest_type', 'dish_liked', 'cuisines', 'approx_costfor_two_people', 'reviews_list', 'menu_item', 'listed_intype', 'listed_incity']


In [7]:
cost_col = "approx_costfor_two_people"

df_clean[cost_col] = (
    df_clean[cost_col]
    .str.replace(",", "", regex=False)
    .str.strip()
)

df_clean[cost_col] = pd.to_numeric(
    df_clean[cost_col],
    errors="coerce"
)

print(df_clean[cost_col].dtype)
print(df_clean[cost_col].describe())
print("\nMissing cost values:", df_clean[cost_col].isna().sum())

float64
count    51371.000000
mean       555.431566
std        438.850728
min         40.000000
25%        300.000000
50%        400.000000
75%        650.000000
max       6000.000000
Name: approx_costfor_two_people, dtype: float64

Missing cost values: 346


In [8]:
missing_pct = (
    df_clean.isnull().mean() * 100
).sort_values(ascending=False)

print(missing_pct[missing_pct > 0])

dish_liked                   54.291626
rate                         19.436549
approx_costfor_two_people     0.669026
rest_type                     0.438927
cuisines                      0.087012
location                      0.040606
dtype: float64


Why dish_liked gets dropped

54.3% missing is way too much.

In [9]:
df_clean["approx_costfor_two_people"] = (
    df_clean["approx_costfor_two_people"]
    .fillna(df_clean["approx_costfor_two_people"].median())
)

# 3. Fill missing restaurant type with mode
df_clean["rest_type"] = (
    df_clean["rest_type"]
    .fillna(df_clean["rest_type"].mode()[0])
)

# 4. Fill missing cuisines with Unknown
df_clean["cuisines"] = (
    df_clean["cuisines"]
    .fillna("Unknown")
)

# 5. Remove rows where location is missing
df_clean = df_clean.dropna(subset=["location"])

print("Shape after cleaning:", df_clean.shape)

Shape after cleaning: (51696, 15)


In [10]:
print("\nRemaining missing values:")
print(df_clean.isnull().sum()[df_clean.isnull().sum() > 0])


Remaining missing values:
rate          10031
dish_liked    28057
dtype: int64


In [11]:
missing_rate = df_clean[df_clean["rate"].isna()]

print("Missing-rate restaurants:", len(missing_rate))

print("\nVotes statistics:")
print(missing_rate["votes"].describe())

print("\nVotes = 0:")
print((missing_rate["votes"] == 0).sum())

print("\nVotes > 0:")
print((missing_rate["votes"] > 0).sum())

Missing-rate restaurants: 10031

Votes statistics:
count    10031.000000
mean         2.029907
std         54.625239
min          0.000000
25%          0.000000
50%          0.000000
75%          0.000000
max       2508.000000
Name: votes, dtype: float64

Votes = 0:
9987

Votes > 0:
44


In [12]:
text_cols = df_clean.select_dtypes(include="object").columns

for col in text_cols:
    print(f"{col}: {df_clean[col].nunique()} unique values")

address: 11486 unique values
name: 8788 unique values


online_order: 2 unique values
book_table: 2 unique values
location: 93 unique values
rest_type: 93 unique values
dish_liked: 5271 unique values
cuisines: 2724 unique values


C:\Users\USER\AppData\Local\Temp\ipykernel_21032\2871476310.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  text_cols = df_clean.select_dtypes(include="object").columns


reviews_list: 22513 unique values
menu_item: 9098 unique values
listed_intype: 7 unique values
listed_incity: 30 unique values


##  Remove High-Cardinality and Non-Predictive Columns

Several columns contain identifiers, highly specific text, or information that is not directly relevant to the locality-based business potential model.

The following columns are removed:

- `name` — restaurant identifier
- `address` — detailed address; locality information is represented by `location`
- `reviews_list` — raw review text is outside the scope of the current model
- `menu_item` — highly variable menu information

The remaining columns contain more generalizable business, location, pricing, and performance characteristics.

In [13]:
columns_to_drop = [
    "name",
    "address",
    "reviews_list",
    "menu_item"
]

df_clean.drop(
    columns=columns_to_drop,
    inplace=True
)

print("Shape:", df_clean.shape)
print(df_clean.columns.tolist())

Shape: (51696, 11)
['online_order', 'book_table', 'rate', 'votes', 'location', 'rest_type', 'dish_liked', 'cuisines', 'approx_costfor_two_people', 'listed_intype', 'listed_incity']


In [14]:
text_cols = df_clean.select_dtypes(include="str").columns

for col in text_cols:
    df_clean[col] = df_clean[col].str.strip()

In [15]:
for col in text_cols:
    print(f"{col}: {df_clean[col].nunique()} unique values")

online_order: 2 unique values
book_table: 2 unique values
location: 93 unique values
rest_type: 93 unique values
dish_liked: 5271 unique values
cuisines: 2724 unique values
listed_intype: 7 unique values
listed_incity: 30 unique values


Final Data Validation

The cleaned dataset is validated to ensure that:

- No duplicate rows remain
- Important columns have the expected data types
- Missing values have been handled appropriately
- Numerical variables are stored in numerical format
- Categorical variables are standardized

In [16]:
print("Final shape:", df_clean.shape)
print("Duplicate rows:", df_clean.duplicated().sum())

Final shape: (51696, 11)
Duplicate rows: 432


In [17]:
print(df_clean.dtypes)

online_order                     str
book_table                       str
rate                         float64
votes                          int64
location                         str
rest_type                        str
dish_liked                       str
cuisines                         str
approx_costfor_two_people    float64
listed_intype                    str
listed_incity                    str
dtype: object


In [18]:
missing = df_clean.isnull().sum()

print(missing[missing > 0])

rate          10031
dish_liked    28057
dtype: int64


In [19]:
display(df_clean.head())
display(df_clean.describe(include="all").T)

,online_order,book_table,rate,votes,location,rest_type,dish_liked,cuisines,approx_costfor_two_people,listed_intype,listed_incity
0,Yes,Yes,4.1,775,Banashankari,Casual Dining,"Pasta, Lunch Buffet, Masala Papad, Paneer Laja...","North Indian, Mughlai, Chinese",800.0,Buffet,Banashankari
1,Yes,No,4.1,787,Banashankari,Casual Dining,"Momos, Lunch Buffet, Chocolate Nirvana, Thai G...","Chinese, North Indian, Thai",800.0,Buffet,Banashankari
2,Yes,No,3.8,918,Banashankari,"Cafe, Casual Dining","Churros, Cannelloni, Minestrone Soup, Hot Choc...","Cafe, Mexican, Italian",800.0,Buffet,Banashankari
3,No,No,3.7,88,Banashankari,Quick Bites,Masala Dosa,"South Indian, North Indian",300.0,Buffet,Banashankari
4,No,No,3.8,166,Basavanagudi,Casual Dining,"Panipuri, Gol Gappe","North Indian, Rajasthani",600.0,Buffet,Banashankari


,count,unique,top,freq,mean,std,min,25%,50%,75%,max
online_order,51696,2,Yes,30444,NaN,NaN,NaN,NaN,NaN,NaN,NaN
book_table,51696,2,No,45247,NaN,NaN,NaN,NaN,NaN,NaN,NaN
rate,41665.0,NaN,NaN,NaN,3.700449,0.440513,1.8,3.4,3.7,4.0,4.9
votes,51696.0,NaN,NaN,NaN,283.812771,803.981766,0.0,7.0,41.0,198.0,16832.0
location,51696,93,BTM,5124,NaN,NaN,NaN,NaN,NaN,NaN,NaN
rest_type,51696,93,Quick Bites,19338,NaN,NaN,NaN,NaN,NaN,NaN,NaN
dish_liked,23639,5271,Biryani,182,NaN,NaN,NaN,NaN,NaN,NaN,NaN
cuisines,51696,2724,North Indian,2913,NaN,NaN,NaN,NaN,NaN,NaN,NaN
approx_costfor_two_people,51696.0,NaN,NaN,NaN,554.454407,437.641522,40.0,300.0,400.0,650.0,6000.0
listed_intype,51696,7,Delivery,25933,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [20]:
df_clean.to_csv(
    "../data/processed/zomato_cleaned.csv",
    index=False
)

print("Cleaned dataset saved successfully!")

Cleaned dataset saved successfully!


Conclusion

The raw Zomato dataset was successfully cleaned and prepared for further analysis.

### Cleaning Summary

- Loaded the raw Zomato dataset containing **51,717 records and 17 columns**
- Standardized column names
- Removed non-predictive fields such as URL and phone number
- Removed highly incomplete `dish_liked` column
- Cleaned and converted restaurant ratings into numerical values
- Converted approximate cost into numerical format
- Handled missing values in location, restaurant type, cuisines, and cost
- Preserved missing ratings as `NaN` because they represent restaurants without an established rating rather than a rating of zero
- Removed non-generalizable fields such as restaurant name, detailed address, raw reviews, and menu items
- Standardized whitespace in categorical variables
- Validated data types and missing values
- Verified the cleaned dataset

### Final Dataset

The resulting dataset contains:

- **51,696 rows**
- **10 columns**
- **No exact duplicate records from the original dataset**
- Missing values only in the `rate` column

### Output

The cleaned dataset has been saved as:

`data/processed/zomato_cleaned.csv`

This processed dataset will be used in the next stage of the project for **Exploratory Data Analysis (EDA), feature engineering, and defining the Business Performance / Success Target**.

---

**Status: Data Cleaning Completed **